# Langchain Agent

## load model

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 1. Load variables from the .env file
load_dotenv()
OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")



llm1 = init_chat_model("google_genai:gemini-2.5-flash")
response = llm1.invoke("What is the color of the sky answer in one word?")
print(response.content)


llm2 = init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com")
response = llm2.invoke("What is the color of the sky answer in one word?")
print(response.content)


Blue
Blue.


## agent

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7
from dataclasses import dataclass

config = {"configurable": {"thread_id": str(uuid7())}}

@dataclass
class Context:
    user_id: str

class Answer(BaseModel):
    summary: str
    confidence: float

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


tools = [search]
agent = create_agent(llm2, 
                    tools=tools,
                    system_prompt="you have to reply in only one line",
                    response_format=Answer,
                    name="my_assistant",
                    checkpointer=InMemorySaver(),
                    context_schema=Context
                      )


result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is life?"}]},
    config=config, context=Context(user_id="user-123")
)

## use of context

Context
Context provides immutable configuration data that is passed at invocation time. Use it for user IDs, session details, or application-specific settings that shouldn’t change during a conversation.

In [4]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.uuid import uuid7



USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder: {user['name']}\n"
            f"Type: {user['account_type']}\n"
            f"Balance: ${user['balance']}"
        )
    return "User not found"



agent = create_agent(
    llm2,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=UserContext(user_id="user123"),
)

In [7]:
result

{'messages': [HumanMessage(content="What's my current balance?", additional_kwargs={}, response_metadata={}, id='0ea0e0d0-fe7b-4543-ae39-40fcb4d70102'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-17T09:43:30.455925044Z', 'done': True, 'done_reason': 'stop', 'total_duration': 876934184, 'load_duration': None, 'prompt_eval_count': 253, 'prompt_eval_duration': None, 'eval_count': 40, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019ed4ff-4423-74c1-a0d7-03e884880bc2-0', tool_calls=[{'name': 'get_account_info', 'args': {}, 'id': '7aab9d06-d838-471c-a5a9-3af68a2a51cf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 253, 'output_tokens': 40, 'total_tokens': 293}),
  ToolMessage(content='Account holder: Alice Johnson\nType: Premium\nBalance: $5000', name='get_account_info', id='86c65682-4968-493b-9969-059c3a83

In [9]:
result["messages"][-1].content

'Your current balance is **$5000**.'

## context in langraph

In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph
from dataclasses import dataclass
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:  # (1)!
    user_id: str

class State(TypedDict, total=False):
    response: str

store = InMemoryStore()  # (2)!
store.put(("users",), "user_123", {"name": "Alice"})

def personalized_greeting(state: State, runtime: Runtime[Context]) -> State:
    '''Generate personalized greeting using runtime context and store.'''
    user_id = runtime.context.user_id  # (3)!
    name = "unknown_user"
    if runtime.store:
        if memory := runtime.store.get(("users",), user_id):
            name = memory.value["name"]

    response = f"Hello {name}! Nice to see you again."
    return {"response": response}

graph = (
    StateGraph(state_schema=State, context_schema=Context)
    .add_node("personalized_greeting", personalized_greeting)
    .set_entry_point("personalized_greeting")
    .set_finish_point("personalized_greeting")
    .compile(store=store)
)

result = graph.invoke({}, context=Context(user_id="user_123"))
print(result)
# > {'response': 'Hello Alice! Nice to see you again.'}

{'response': 'Hello Alice! Nice to see you again.'}


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 1. Define the exact structure you want the final answer to have
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="Must be 'Positive', 'Negative', or 'Neutral'")
    score: int = Field(description="A satisfaction rating from 1 to 10")
    summary: str = Field(description="A one-sentence summary of the user's issue")

# 2. Instantiate the model
# llm = init_chat_model("openai:gpt-4o", temperature=0)

# 3. Create the agent and pass the schema to response_format
agent = create_agent(
    model=llm2,
    response_format=ReviewAnalysis # <-- Forces the final answer into this schema
)

# 4. Invoke the agent with a customer complaint
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed."}]}
)



In [13]:
result

{'messages': [HumanMessage(content='I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed.', additional_kwargs={}, response_metadata={}, id='1004c0f6-b470-4766-8dab-76a7a6f5b569'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-17T09:57:55.700902214Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6004522349, 'load_duration': None, 'prompt_eval_count': 378, 'prompt_eval_duration': None, 'eval_count': 204, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019ed50c-640f-7242-afad-237537940011-0', tool_calls=[{'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': 'The sole ripped off completely within two hours of wearing, despite the shoes looking amazing.'}, 'id': '562d1ee6-0f4b-463e-b0f3-d7f39f00da8e', 'type': 'tool_call'}], in

In [14]:
# ==========================================================
# 5. Output Results
# ==========================================================
# When using response_format, the structured data lives inside result["response"]
analysis = result["structured_response"]

print(type(analysis))  # This will be <class '__main__.ReviewAnalysis'>
print("Sentiment:", analysis.sentiment)
print("Score:", analysis.score)
print("Summary:", analysis.summary)

<class '__main__.ReviewAnalysis'>
Sentiment: Negative
Score: 2
Summary: The sole ripped off completely within two hours of wearing, despite the shoes looking amazing.


In [15]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 1. Define the exact structure you want the final answer to have
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="Must be 'Positive', 'Negative', or 'Neutral'")
    score: int = Field(description="A satisfaction rating from 1 to 10")
    summary: str = Field(description="A one-sentence summary of the user's issue")

# 2. Instantiate the model
# llm = init_chat_model("openai:gpt-4o", temperature=0)

# 3. Create the agent and pass the schema to response_format
agent = create_agent(
    model=llm1,
    response_format=ReviewAnalysis # <-- Forces the final answer into this schema
)

# 4. Invoke the agent with a customer complaint
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed."}]}
)



In [16]:
result

{'messages': [HumanMessage(content='I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed.', additional_kwargs={}, response_metadata={}, id='b043199d-3d18-43b1-95c4-ff8f60cdc189'),
  AIMessage(content='{"sentiment": "Negative", "score": 1, "summary": "The sole of the new shoes ripped off within two hours of walking."}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ed515-7aa2-7982-a67f-e3036318b923-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 176, 'total_tokens': 201, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 145}})],
 'structured_response': ReviewAnalysis(sentiment='Negative', score=1, summary='The sole of the new shoes ripped off within two hours of walking.')}

[]

In [28]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 1. Define the exact structure you want the final answer to have
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="Must be 'Positive', 'Negative', or 'Neutral'")
    score: int = Field(description="A satisfaction rating from 1 to 10")
    summary: str = Field(description="A one-sentence summary of the user's issue")

# 2. Instantiate the model
# llm = init_chat_model("openai:gpt-4o", temperature=0)

# 3. Create the agent and pass the schema to response_format
agent = create_agent(
    model=llm2,
    response_format=ReviewAnalysis ,# <-- Forces the final answer into this schema
    debug=True
)

# 4. Invoke the agent with a customer complaint
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed."}]}
)



[values] {'messages': [HumanMessage(content='I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed.', additional_kwargs={}, response_metadata={}, id='8b5533e9-6cf1-4cc4-81ef-13bbed1d36e5')]}


[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-17T10:31:12.88198545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5102407383, 'load_duration': None, 'prompt_eval_count': 378, 'prompt_eval_duration': None, 'eval_count': 202, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019ed52a-e11a-7e81-a43a-22317ac0d11e-0', tool_calls=[{'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': 'Sole ripped off within two hours of walking despite the shoes looking amazing.'}, 'id': '8e093851-1ac6-404f-9494-12752c6389e9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 378, 'output_tokens': 202, 'total_tokens': 580}), ToolMessage(content="Returning structured response: sentiment='Negative' score=2 summary='Sole ripped off within two hours of walking despit

In [25]:
result

{'messages': [HumanMessage(content='I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed.', additional_kwargs={}, response_metadata={}, id='e56c0a60-3c35-4609-9c0a-3cab28849453'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-17T10:13:01.522686754Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3077312677, 'load_duration': None, 'prompt_eval_count': 378, 'prompt_eval_duration': None, 'eval_count': 242, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019ed51a-41c2-77d0-8333-20b450fb1077-0', tool_calls=[{'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}, 'id': 'f6310806-9372-4988-b93a-40a809aac592', 'type': 'tool_call'}], 

In [26]:
result['messages']

[HumanMessage(content='I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed.', additional_kwargs={}, response_metadata={}, id='e56c0a60-3c35-4609-9c0a-3cab28849453'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-17T10:13:01.522686754Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3077312677, 'load_duration': None, 'prompt_eval_count': 378, 'prompt_eval_duration': None, 'eval_count': 242, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019ed51a-41c2-77d0-8333-20b450fb1077-0', tool_calls=[{'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}, 'id': 'f6310806-9372-4988-b93a-40a809aac592', 'type': 'tool_call'}], invalid_tool_c

In [27]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# ==========================================================
# 4. STEP-BY-STEP VISUALIZATION PRINTING
# ==========================================================
print("\n=== STEP-BY-STEP LLM INTERACTION TIMELINE ===\n")

for i, msg in enumerate(result["messages"], start=1):
    print(f"--- [STEP {i}] {type(msg).__name__} ---")
    
    # If it's the User Input
    if msg.type == "human":
        print(f"Content: \"{msg.content}\"\n")
        
    # If it's the LLM deciding to return structured data
    elif msg.type == "ai":
        print(f"Text Content: \"{msg.content}\" (Empty because it chose to call a tool)")
        print(f"Tool Calls Detected: {msg.tool_calls}\n")
        
    # If it's the graph closing the tool loop
    elif msg.type == "tool":
        print(f"Matched Tool Call ID: {msg.tool_call_id}")
        print(f"Internal Status Msg: \"{msg.content}\"\n")

print("=============================================")
print("=== [FINAL STEP] PARSED STRUCTURED RESPONSE ===")
print("=============================================")
# LangChain extracts the final tool arguments and converts them back to your Pydantic object here:
print(result["structured_response"])


=== STEP-BY-STEP LLM INTERACTION TIMELINE ===

--- [STEP 1] HumanMessage ---
Content: "I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed."

--- [STEP 2] AIMessage ---
Text Content: "" (Empty because it chose to call a tool)
Tool Calls Detected: [{'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}, 'id': 'f6310806-9372-4988-b93a-40a809aac592', 'type': 'tool_call'}]

--- [STEP 3] ToolMessage ---
Matched Tool Call ID: f6310806-9372-4988-b93a-40a809aac592
Internal Status Msg: "Returning structured response: sentiment='Negative' score=2 summary="The shoes' sole detached completely within two hours of use despite their attractive appearance.""

=== [FINAL STEP] PARSED STRUCTURED RESPONSE ===
sentiment='Negative' score=2 summary="The shoes' sole detached completely within two hours 

To visualize the exact step-by-step lifecycle of how your agent handled the interaction with Nemotron, we can break down your raw response object into a chronological timeline of data states.

---

### Step-by-Step Visualization

```
[1. User Input] ──> HumanMessage: "I bought this shoes..."
       │
       ▼
[2. Agent Node] ──> Nemotron (via Ollama) processes bound schema
       │
       ▼
[3. LLM Response] ──> AIMessage (content="", tool_calls=[ReviewAnalysis])
       │
       ▼
[4. Execution]   ──> ToolMessage: "Returning structured response..."
       │
       ▼
[5. Validation]  ──> LangChain converts parameters to ReviewAnalysis object
       │
       ▼
[6. Final Output] ──> 'structured_response': ReviewAnalysis(sentiment='Negative'...)

```

---

### Step-by-Step State Inspection

Here is the precise structural translation of your specific interaction, step-by-step:

### Step 1: The Input State (`HumanMessage`)

The user initializes the execution by sending text into the graph. This is appended as a standard `HumanMessage`.

* **Payload Type:** `HumanMessage`
* **Content:** `"I bought this shoes yesterday. They look amazing but the sole ripped completely off within two hours of walking. Very disappointed."`
* **Metadata Tracking:** A unique runtime ID (`1004c0f6-...`) is generated to track this input message through the system.

### Step 2: The Core Decision State (`AIMessage`)

The graph passes the message stream to Nemotron along with the `tool_choice` argument forcing the `ReviewAnalysis` layout.

* **Payload Type:** `AIMessage`
* **Content:** `''` *(The text is empty because the model directly invoked a programmatic schema rather than answering conversationally).*
* **The Magic Block (`tool_calls`):** ```python
[{
'name': 'ReviewAnalysis',
'args': {
'sentiment': 'Negative',
'score': 2,
'summary': 'The sole ripped off completely within two hours of wearing, despite the shoes looking amazing.'
},
'id': '562d1ee6-0f4b-463e-b0f3-d7f39f00da8e',
'type': 'tool_call'
}]
```

```


* **Performance Metadata (`response_metadata`):** The engine recorded that this model execution on your machine took roughly 6 seconds (`total_duration: 6004522349` nanoseconds) and consumed `582` total tokens.

### Step 3: Closing the Loop (`ToolMessage`)

Because the agent generated an active tool call block internally, LangGraph must transition to a tools node to close the loop. LangChain automatically handles this by inserting a programmatic acknowledgment:

* **Payload Type:** `ToolMessage`
* **`tool_call_id`:** `562d1ee6-0f4b-463e-b0f3-d7f39f00da8e` *(This matches the exact execution ID given by the AIMessage above, locking the pair together).*
* **Content:** `"Returning structured response: sentiment='Negative' score=2 summary='...'"`

### Step 4: Final State Extraction (`structured_response`)

Once the graph loop finishes processing the text exchanges, it reads the tool arguments out of Step 2, converts the raw JSON dictionary back into a structured Python object using your Pydantic schema, and updates your final graph output.

* **Payload Key:** `'structured_response'`
* **Final Form:** `ReviewAnalysis(sentiment='Negative', score=2, summary='The sole ripped off completely...')`



```python
[{
'name': 'ReviewAnalysis',
'args': {
'sentiment': 'Negative',
'score': 2,
'summary': 'The sole ripped off completely within two hours of wearing, despite the shoes looking amazing.'
},
'id': '562d1ee6-0f4b-463e-b0f3-d7f39f00da8e',
'type': 'tool_call'
}]
```

In [ ]:
import json
from pydantic import BaseModel, Field

# 1. Your exact defined Pydantic model
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="Must be 'Positive', 'Negative', or 'Neutral'")
    score: int = Field(description="A satisfaction rating from 1 to 10")
    summary: str = Field(description="A brief keyword summary")

# ==========================================================
# THE EXACT METHOD LANGCHAIN CALLS TO GET THE SCHEMA
# ==========================================================
# This built-in Pydantic v2 function extracts the structural data map
raw_schema = ReviewAnalysis.model_json_schema()

# LangChain packages it into a named structural object format for the API
llm_payload_schema = {
    "name": "ReviewAnalysis",
    "description": "Generated tool definition for structured output tracking.",
    "parameters": raw_schema
}

# Print the final result in clean JSON format
print(json.dumps(llm_payload_schema, indent=2))

{
  "name": "ReviewAnalysis",
  "description": "Extract structured keywords, satisfaction score, and overall sentiment from the query.",
  "parameters": {
    "properties": {
      "sentiment": {
        "description": "Must be 'Positive', 'Negative', or 'Neutral'",
        "title": "Sentiment",
        "type": "string"
      },
      "score": {
        "description": "A satisfaction rating from 1 to 10",
        "title": "Score",
        "type": "integer"
      },
      "summary": {
        "description": "A brief keyword summary",
        "title": "Summary",
        "type": "string"
      }
    },
    "required": [
      "sentiment",
      "score",
      "summary"
    ],
    "title": "ReviewAnalysis",
    "type": "object"
  }
}


In [36]:
extracted_pydantic_values = {'name': 'ReviewAnalysis', 'args': {'sentiment': 'Negative', 'score': 2, 'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}}
extracted_pydantic_values

{'name': 'ReviewAnalysis',
 'args': {'sentiment': 'Negative',
  'score': 2,
  'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}}

In [44]:
# How LangChain populates your state history dynamically
from langchain.messages import ToolMessage, AIMessage, SystemMessage

content_text = f"Returning structured response: {extracted_pydantic_values}"

# It constructs the ToolMessage
tracking_message = ToolMessage(
    content=content_text, 
    name="ReviewAnalysis", 
    tool_call_id="562d1ee6-0f4b-463e-b0f3-d7f39f00da8e"
)

In [45]:
tracking_message

ToolMessage(content='Returning structured response: {\'name\': \'ReviewAnalysis\', \'args\': {\'sentiment\': \'Negative\', \'score\': 2, \'summary\': "The shoes\' sole detached completely within two hours of use despite their attractive appearance."}}', name='ReviewAnalysis', tool_call_id='562d1ee6-0f4b-463e-b0f3-d7f39f00da8e')

In [47]:
content_text

'Returning structured response: {\'name\': \'ReviewAnalysis\', \'args\': {\'sentiment\': \'Negative\', \'score\': 2, \'summary\': "The shoes\' sole detached completely within two hours of use despite their attractive appearance."}}'

In [50]:
extracted_pydantic_values['args']

{'sentiment': 'Negative',
 'score': 2,
 'summary': "The shoes' sole detached completely within two hours of use despite their attractive appearance."}

In [52]:
# Validates and instantiates the Pydantic class object
final_pydantic_object = ReviewAnalysis.model_validate(extracted_pydantic_values['args'])

In [53]:
final_pydantic_object

ReviewAnalysis(sentiment='Negative', score=2, summary="The shoes' sole detached completely within two hours of use despite their attractive appearance.")

In [54]:
## pure with google gemini

In [55]:
import os
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# 1. Define your desired output structure using Pydantic
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="Must be 'Positive', 'Negative', or 'Neutral'")
    score: int = Field(description="A satisfaction rating from 1 to 10")
    summary: str = Field(description="A brief one-sentence summary of the main issue")

# 2. Initialize the native Google GenAI Client
# It automatically picks up the GEMINI_API_KEY environment variable
client = genai.Client()

# 3. Configure the Gemini structural constraint parameters
config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=ReviewAnalysis,          # Direct Pydantic native support!
    temperature=0.0                         # Keep it deterministic for structured tasks
)

# 4. Execute the call using the gemini-2.5-flash model
user_prompt = (
    "I bought these shoes yesterday. They look amazing but the sole ripped "
    "completely off within two hours of walking. Very disappointed."
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_prompt,
    config=config
)

# ==========================================================
# 5. Extract and Validate the Output
# ==========================================================
# Gemini returns a validated JSON string inside response.text
print("Raw JSON String from Gemini API:")
print(response.text)
print("-" * 40)

# Parse it back into your local Python Pydantic object
structured_output = ReviewAnalysis.model_validate_json(response.text)

print("Parsed Python Object properties:")
print(f"Sentiment: {structured_output.sentiment}")
print(f"Score:     {structured_output.score}")
print(f"Summary:   {structured_output.summary}")

Raw JSON String from Gemini API:
{"sentiment":"Negative","score":1,"summary":"The shoe sole ripped off within two hours of walking after purchase."}
----------------------------------------
Parsed Python Object properties:
Sentiment: Negative
Score:     1
Summary:   The shoe sole ripped off within two hours of walking after purchase.


In [56]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='{"sentiment":"Negative","score":1,"summary":"The shoe sole ripped off within two hours of walking after purchase."}'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  parsed=ReviewAnalysis(
    score=1,
    sentiment='Negative',
    summary='The shoe sole ripped off within two hours of walking after purchase.'
  ),
  response_id='2XsyaqHUJqu2juMPrvCbkAY',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=25,
    prompt_token_count=25,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=25
      ),
    ],
    thoughts_token_count=138,
    tota

In [57]:
print(response.text)

{"sentiment":"Negative","score":1,"summary":"The shoe sole ripped off within two hours of walking after purchase."}


In [59]:
analysis_obj = response.parsed
analysis_obj

ReviewAnalysis(sentiment='Negative', score=1, summary='The shoe sole ripped off within two hours of walking after purchase.')

## Emotion classifier

In [61]:
from typing import Literal
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

class EmotionClassifier(BaseModel):
    emotion:Literal['Happy','Sad','Angry']=Field(description="The primary emotion detected in the text.")
    confidence_score:int=Field(description="Confidence score from 1 to 10")
    reason:str=Field(description="A short one-sentence explanation of why this emotion was chosen.")

llm=init_chat_model("google_genai:gemini-2.5-flash",temperature=0)

agent=create_agent(model=llm,response_format=EmotionClassifier)

user_input="I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!"
result=agent.invoke({"messages":[{"role":"user","content":user_input}]})
parsed_output=result["structured_response"]
print("=== EMOTION CLASSIFICATION RESULT ===")
print(f"Detected Emotion: {parsed_output.emotion}")
print(f"Confidence:       {parsed_output.confidence_score}%")
print(f"Reasoning:        {parsed_output.reason}")

=== EMOTION CLASSIFICATION RESULT ===
Detected Emotion: Angry
Confidence:       9%
Reasoning:        The user expresses frustration and anger over a three-week delay in their order and a lack of response to their emails.


In [62]:
result

{'messages': [HumanMessage(content='I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!', additional_kwargs={}, response_metadata={}, id='e4f39b54-fe21-4b6d-a4ed-612abb36f9a8'),
  AIMessage(content='{"emotion": "Angry", "confidence_score": 9, "reason": "The user expresses frustration and anger over a three-week delay in their order and a lack of response to their emails."}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ed548-f930-7a80-bd7a-aa0adb8452eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 164, 'total_tokens': 189, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 121}})],
 'structured_response': EmotionClassifier(emotion='Angry', confidence_score=9, reason='The user expresses frustration and anger over a thre

## using langraph

In [64]:
from typing import Literal
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import Annotated,TypedDict

class EmotionClassifier(BaseModel):
    emotion:Literal['Happy','Sad','Angry']=Field(description="The primary emotion detected in the text.")
    confidence_score:int=Field(description="Confidence score from 1 to 10")
    reason:str=Field(description="A short one-sentence explanation of why this emotion was chosen.")

class State(TypedDict):
    messages:Annotated[list,add_messages]
    structured_response:EmotionClassifier

llm=init_chat_model("google_genai:gemini-2.5-flash",temperature=0)

structured_llm=llm.with_structured_output(EmotionClassifier)

def classify_emotion(state:State):
    response=structured_llm.invoke(state["messages"])
    return {"structured_response":response}

builder=StateGraph(State)
builder.add_node("classifier",classify_emotion)
builder.add_edge(START,"classifier")
builder.add_edge("classifier",END)
graph=builder.compile()

user_input="I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!"
result=graph.invoke({"messages":[HumanMessage(content=user_input)]})


parsed_output=result["structured_response"]
print("=== EMOTION CLASSIFICATION RESULT ===")
print(f"Detected Emotion: {parsed_output.emotion}")
print(f"Confidence:       {parsed_output.confidence_score}%")
print(f"Reasoning:        {parsed_output.reason}")

=== EMOTION CLASSIFICATION RESULT ===
Detected Emotion: Angry
Confidence:       9%
Reasoning:        The user expresses frustration and anger over a three-week delay in their order and a lack of response to their emails.


In [65]:
result

{'messages': [HumanMessage(content='I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!', additional_kwargs={}, response_metadata={}, id='7907ea7a-425f-42c6-8184-2edf259ca1e7')],
 'structured_response': EmotionClassifier(emotion='Angry', confidence_score=9, reason='The user expresses frustration and anger over a three-week delay in their order and a lack of response to their emails.')}

In [68]:
result['messages']

[HumanMessage(content='I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!', additional_kwargs={}, response_metadata={}, id='7907ea7a-425f-42c6-8184-2edf259ca1e7')]

### with nemotron don't has structured output option

In [ ]:
from typing import Literal
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import Annotated,TypedDict

class EmotionClassifier(BaseModel):
    emotion:Literal['Happy','Sad','Angry']=Field(description="The primary emotion detected in the text.")
    confidence_score:int=Field(description="Confidence score from 1 to 10")
    reason:str=Field(description="A short one-sentence explanation of why this emotion was chosen.")

class State(TypedDict):
    messages:Annotated[list,add_messages]
    structured_response:EmotionClassifier

llm=init_chat_model("google_genai:gemini-2.5-flash",temperature=0)

structured_llm=llm2.with_structured_output(EmotionClassifier)

def classify_emotion(state:State):
    response=structured_llm.invoke(state["messages"])
    return {"structured_response":response}

builder=StateGraph(State)
builder.add_node("classifier",classify_emotion)
builder.add_edge(START,"classifier")
builder.add_edge("classifier",END)
graph=builder.compile()

user_input="I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!"
result=graph.invoke({"messages":[HumanMessage(content=user_input)]})


parsed_output=result["structured_response"]
print("=== EMOTION CLASSIFICATION RESULT ===")
print(f"Detected Emotion: {parsed_output.emotion}")
print(f"Confidence:       {parsed_output.confidence_score}%")
print(f"Reasoning:        {parsed_output.reason}")

Here’s what I’d recommend doing **right now** to escalate this effectively:  

### 🔑 Immediate Steps to Take:
1. **Switch Communication Channels (Don’t Rely on Email Alone):**  
   - **Call their customer service line** (if available)—ask for a supervisor *immediately*. Phone calls create urgency emails often lack.  
   - **Try live chat** on their website (if offered)—agents there sometimes have more authority to act.  
   - **Contact them via social media** (Twitter/X, Facebook, Instagram). Public comments or DMs often get faster replies because companies care about their public image. *Example tweet:*  
     > *"@[Company] I’ve waited 3 weeks for order #[Number] with no response to 5+ emails. This is unacceptable. Please DM me to resolve this NOW or I’ll escalate to [BBB/FTC] and share my experience publicly."*  

2. **Document Everything:**  
   - Save all email timestamps, order numbers, and any proof of purchase.  
   - Note dates/times you called or chatted (and who you spoke with, if possible).  

3. **Escalate Beyond Frontline Support:**  
   - Ask for a **manager, supervisor, or customer retention team**—frontline agents often can’t override systemic delays.  
   - If they still don’t respond, **file a dispute** with your payment provider (credit card/PayPal)—they often reverse charges for unresponsive merchants within days.  
   - Report to the **[Better Business Bureau](https://www.bbb.org)** or **[FTC Complaint Assistant](https://reportfraud.ftc.gov/)**—this pressures companies to act.  

### 💡 Why This Happens (and Why It’s Not Your Fault):
- Some companies understaff support or prioritize new sales over existing customers (a shortsighted and damaging practice).  
- Automated systems might have flagged your order incorrectly (e.g., fraud check, inventory error), but **no excuse exists for ghosting you**.  
- You’re not being “difficult”—you’re holding them accountable for a basic service standard: *communicate delays*.  

### 🌟 Key Mindset Shift:
> **You’re not “bothering” them—you’re enforcing your rights as a customer.**  
> A business that ignores you for 3 weeks doesn’t deserve your patience—or your future business. If they won’t fix this now, consider taking your money elsewhere next time (and sharing your experience honestly to help others avoid this).  

You deserve transparency and respect. If you’d like, I can help draft a firm but polite escalation email or social media message—just share the company name (if you’re comfortable) and your order details (blur sensitive info). You’ve got this. 💪  

*P.S. If this is a small business, sometimes a kind-but-firm note acknowledging they might be overwhelmed *while* insisting on a timeline can work—but only if they’ve shown *any* willingness to engage. Silence after 3 weeks? Time for harder tactics.*
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
During task with name 'classifier' and

In [70]:
from typing import Literal,Annotated,TypedDict
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
class EmotionClassifier(BaseModel):
    emotion:Literal['Happy','Sad','Angry']=Field(description="The primary emotion detected in the text.")
    confidence_score:int=Field(description="Confidence score from 1 to 10")
    reason:str=Field(description="A short one-sentence explanation of why this emotion was chosen.")
class State(TypedDict):
    messages:Annotated[list,add_messages]
    structured_response:EmotionClassifier

llm2=init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com",temperature=0)

parser=JsonOutputParser(pydantic_object=EmotionClassifier)

prompt=ChatPromptTemplate.from_messages([("system","You are a helpful assistant. Respond strictly in JSON format matching the schema instructions.\n{format_instructions}"),("placeholder","{messages}")])

chain=prompt|llm2|parser


def classify_emotion(state:State):
    response=chain.invoke({"messages":state["messages"],"format_instructions":parser.get_format_instructions()})
    return {"structured_response":response}

builder=StateGraph(State)
builder.add_node("classifier",classify_emotion)
builder.add_edge(START,"classifier")
builder.add_edge("classifier",END)
graph=builder.compile()
user_input="I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!"
result=graph.invoke({"messages":[HumanMessage(content=user_input)]})
parsed_output=result["structured_response"]
print("=== EMOTION CLASSIFICATION RESULT ===")
print(f"Detected Emotion: {parsed_output['emotion']}")
print(f"Confidence:       {parsed_output['confidence_score']}%")
print(f"Reasoning:        {parsed_output['reason']}")

=== EMOTION CLASSIFICATION RESULT ===
Detected Emotion: Angry
Confidence:       9%
Reasoning:        The user expresses frustration about a delayed order and lack of response, indicating anger.


In [71]:
from typing import Literal,Annotated,TypedDict
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.structured_output import ToolStrategy
class EmotionClassifier(BaseModel):
    emotion:Literal['Happy','Sad','Angry']=Field(description="The primary emotion detected in the text.")
    confidence_score:int=Field(description="Confidence score from 1 to 10")
    reason:str=Field(description="A short one-sentence explanation of why this emotion was chosen.")
llm2=init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com",temperature=0)
agent=create_agent(model=llm2,response_format=ToolStrategy(schema=EmotionClassifier))
user_input="I have been waiting for my order for three weeks and no one is replying to my emails! This is completely unacceptable!"
result=agent.invoke({"messages":[{"role":"user","content":user_input}]})
parsed_output=result["structured_response"]
print("=== EMOTION CLASSIFICATION RESULT ===")
print(f"Detected Emotion: {parsed_output.emotion}")
print(f"Confidence:       {parsed_output.confidence_score}%")
print(f"Reasoning:        {parsed_output.reason}")

=== EMOTION CLASSIFICATION RESULT ===
Detected Emotion: Angry
Confidence:       9%
Reasoning:        User expresses frustration and anger about unaddressed order delay and lack of response to emails, calling it completely unacceptable.
